In [1]:
# ──────────────────────────────────────────────────────────────────────
# Real-Time Credit Card Fraud Detection — ML Pipeline
# Dataset: Kaggle Credit Card Fraud Detection (284,807 transactions, 492 fraud)
# Goal: Train an XGBoost model to detect fraudulent credit card transactions
# ──────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
import joblib


In [ ]:
pip install xgboost       

In [2]:
# Load the dataset
# 284,807 transactions made by European cardholders in September 2013
# Features V1-V28 are PCA-transformed (original features anonymized for privacy)
# Amount: transaction value in euros | Time: seconds elapsed since first transaction
# Class: 0 = normal, 1 = fraud
df = pd.read_csv('creditcard.csv')


In [ ]:
# Basic sanity check — confirm data loaded correctly
print(f"Shape: {df.shape}")


In [ ]:
# Class distribution: how many normal vs fraud transactions
# Expect extreme imbalance — fraud is very rare in real card data
print(f"\nFraud distribution:\n{df['Class'].value_counts()}")


In [ ]:
# Fraud rate as a percentage — this tells us how imbalanced the dataset is
# 0.17% fraud means for every 1 fraud there are ~577 normal transactions
# This imbalance is the main challenge — standard ML models will just predict "normal" for everything
print(f"\nFraud rate: {df['Class'].mean()*100:.3f}%")


In [ ]:
# Preview the raw data
# V1-V28: already normalized by PCA (good scale, no action needed)
# Amount and Time: raw values — will need StandardScaler before modeling
df.head()


In [ ]:
# Isolate fraud transactions to study their characteristics
fraud_df = df[df['Class'] == 1]
print(f"Total fraud transactions: {len(fraud_df)}")
fraud_df.head(10)


In [ ]:
# Visualize the class imbalance and transaction amount distribution
# Left chart: confirms how rare fraud is (0.17% of all transactions)
# Right chart: amount distribution — fraudulent transactions cluster at lower amounts

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Class'].value_counts().plot(kind='pie', ax=axes[0],
    labels=['Normal', 'Fraud'], autopct='%1.2f%%', colors=['steelblue', 'tomato'])
axes[0].set_title('Fraud vs Normal Distribution')

axes[1].hist(df[df['Class']==0]['Amount'], bins=50, alpha=0.6, label='Normal', color='steelblue')
axes[1].hist(df[df['Class']==1]['Amount'], bins=50, alpha=0.6, label='Fraud', color='tomato')
axes[1].set_title('Transaction Amount Distribution')
axes[1].set_xlabel('Amount')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Zoom in to 0-500€ range to see the distribution shape more clearly
# The long tail from high-value transactions was obscuring the pattern

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df[df['Class']==0]['Amount'], bins=50, range=(0, 500), alpha=0.6, label='Normal', color='steelblue')
axes[0].set_title('Normal Transactions (0-500€)')
axes[0].set_xlabel('Amount')
axes[0].legend()

axes[1].hist(df[df['Class']==1]['Amount'], bins=50, range=(0, 500), alpha=0.6, label='Fraud', color='tomato')
axes[1].set_title('Fraud Transactions (0-500€)')
axes[1].set_xlabel('Amount')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Summary statistics: max and mean amounts by class
# Fraudsters tend to avoid very large transactions to stay under the radar
print(f"Normal max amount:  ${df[df['Class']==0]['Amount'].max():.2f}")
print(f"Fraud max amount:   ${df[df['Class']==1]['Amount'].max():.2f}")
print(f"Normal mean amount: ${df[df['Class']==0]['Amount'].mean():.2f}")
print(f"Fraud mean amount:  ${df[df['Class']==1]['Amount'].mean():.2f}")


In [ ]:
# Distribution of the top PCA features by fraud correlation
# V14, V17, V12 show the clearest separation between fraud (red) and normal (blue)
# Even though we don't know what V14 represents (it's anonymized), the signal is strong
# — fraud cases cluster in a narrow band while normal transactions are spread wide

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, feature in enumerate(['V14', 'V17', 'V12']):
    axes[i].hist(df[df['Class']==0][feature], bins=50, alpha=0.6, label='Normal', color='steelblue')
    axes[i].hist(df[df['Class']==1][feature], bins=50, alpha=0.6, label='Fraud', color='tomato')
    axes[i].set_title(f'{feature} Distribution')
    axes[i].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Find which features have the strongest correlation with fraud
# Higher value = better predictor of fraud
# V14, V17, V12 consistently rank at the top — these drive the model the most
correlations = df.corr()['Class'].drop('Class').abs().sort_values(ascending=False)
print(correlations.head(10))


In [ ]:
# Feature Engineering — Normalize Amount and Time
# V1-V28 are already PCA-transformed (good scale), but Amount and Time are raw values.
# StandardScaler converts them to mean=0, std=1 so they're on the same scale as V1-V28.
#
# Important: We scale on the full dataset here only for EDA visualization (cells below).
# For model training, we re-apply scaling AFTER the train/test split — this is the
# correct ML practice. Scaling before splitting means the test set's statistics
# influence the scaler, which is a form of data leakage.

# Keep original Amount and Time for correct post-split scaling later
df_original_amount_time = df[['Amount', 'Time']].copy()

scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])
df = df.drop(['Amount', 'Time'], axis=1)

print("Feature engineering done ✅")
print(df.shape)


In [ ]:
# Confirm the final column list after feature engineering
# 28 PCA features + Amount_scaled + Time_scaled + Class = 31 columns
df.columns.tolist()


In [ ]:
# Preview the processed dataframe — all features are now on a comparable scale
df.head()


In [ ]:
# Verify scaling worked correctly — StandardScaler output should be centered near 0
print(f"Amount_scaled: min={df['Amount_scaled'].min():.2f}, max={df['Amount_scaled'].max():.2f}")
print(f"Time_scaled:   min={df['Time_scaled'].min():.2f}, max={df['Time_scaled'].max():.2f}")


In [ ]:
# Train/Test Split
# stratify=y: both train and test sets maintain the same 0.17% fraud ratio
# 80% train (~228K rows), 20% test (~57K rows)

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Correct scaling: fit scaler ONLY on training data, then transform both sets
# This prevents data leakage — test statistics should have zero influence on training
# We use the original raw Amount/Time values (saved above) for clean re-scaling
scaler_final = StandardScaler()

amount_time_train = df_original_amount_time.loc[X_train.index]
amount_time_test  = df_original_amount_time.loc[X_test.index]

X_train = X_train.copy()
X_test  = X_test.copy()

X_train[['Amount_scaled', 'Time_scaled']] = scaler_final.fit_transform(amount_time_train)
X_test[['Amount_scaled', 'Time_scaled']]  = scaler_final.transform(amount_time_test)

# Save scaler alongside model — production inference must use the same scaling parameters
joblib.dump(scaler_final, 'scaler.pkl')
print("Scaler saved as scaler.pkl ✅")

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Train fraud count: {y_train.sum()}")
print(f"Test fraud count:  {y_test.sum()}")


In [ ]:
  # Base Model (Without SMOTE)                                                                                                                                             
  # Before applying SMOTE, we train a baseline XGBoost model on the imbalanced dataset.
  # This gives us a reference point to compare against the SMOTE-balanced model.                                                                                           
  # The model is expected to miss most fraud cases due to the class imbalance.
                                                                                                                                                                           
  from xgboost import XGBClassifier
  from sklearn.metrics import classification_report                                                                                                                        
                  
  base_model = XGBClassifier(random_state=42, eval_metric='logloss')                                                                                                       
  base_model.fit(X_train, y_train)
                                                                                                                                                                           
  y_pred = base_model.predict(X_test)
  print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

In [ ]:
  # Results (Without SMOTE):                                                                                                                                               
  # Precision: 0.92 - 92% of predicted frauds were actually fraud
  # Recall: 0.81 - model caught 81% of real frauds, missed 19%                                                                                                             
  # F1: 0.86      
  # This is our baseline. We will try to improve recall with SMOTE.    

In [ ]:
pip install imbalanced-learn   

In [ ]:
                                                                                                                                                                           
  # SMOTE Application                                                                                                                                                      
  # We oversample the minority class (fraud) to balance the training data.                                                                                                 
  # This does NOT train the model yet — it only prepares the balanced dataset. 
  # Before: 394 fraud | After: 227,451 fraud (synthetic samples added)  

  from imblearn.over_sampling import SMOTE                                                                                                                                 
                                                                                                                                                                           
  smote = SMOTE(random_state=42)
  X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
                                                                                                                                                                           
  print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
  print(f"After SMOTE: {y_train_resampled.value_counts().to_dict()}") 

In [ ]:
  # Train XGBoost on SMOTE-balanced data                                                                                                                                   
  # Comparing results with baseline to see if recall improves on fraud detection                                                                                           
                                                                                                                                                                           
  model = XGBClassifier(random_state=42, eval_metric='logloss')                                                                                                            
  model.fit(X_train_resampled, y_train_resampled)                                                                                                                          
                                                                                                                                                                           
  y_pred_smote = model.predict(X_test)
  print(classification_report(y_test, y_pred_smote, target_names=['Normal', 'Fraud']))

In [ ]:
  # Results Comparison (Base vs SMOTE)                                                                                                                                     
  # Base model (no SMOTE): Precision=0.92, Recall=0.81, F1=0.86
  # SMOTE model:           Precision=0.73, Recall=0.89, F1=0.80                                                                                                            
  #                                                                                                                                                                        
  # SMOTE improved recall (catches more fraud) but reduced precision (more false alarms)                                                                                   
  # Trade-off: missing real fraud is worse than false alarms in banking                                                                                                    
  # Next step: threshold tuning to find the optimal balance      

In [ ]:
  # Precision-Recall Curve & Optimal Threshold                                                                                                                             
  # We plot the PR curve to find the threshold that maximizes F1 score                                                                                                     
  # This helps us balance precision and recall optimally                                                                                                                   
                  
  from sklearn.metrics import precision_recall_curve                                                                                                                       
  import numpy as np
                                                                                                                                                                           
  y_scores = model.predict_proba(X_test)[:, 1]                                                                                                                             
  precisions, recalls, thresholds = precision_recall_curve(y_test, y_scores)
                                                                                                                                                                           
  f1_scores = 2 * (precisions * recalls) / (precisions + recalls)
  best_threshold = thresholds[np.argmax(f1_scores)]
                                                                                                                                                                           
  print(f"Best threshold: {best_threshold:.4f}")
  print(f"Best F1: {np.max(f1_scores):.4f}")                                                                                                                               
                                                                                                                                                                           
  plt.figure(figsize=(8, 5))
  plt.plot(thresholds, precisions[:-1], label='Precision', color='steelblue')                                                                                              
  plt.plot(thresholds, recalls[:-1], label='Recall', color='tomato')                                                                                                       
  plt.plot(thresholds, f1_scores[:-1], label='F1', color='green')
  plt.axvline(best_threshold, linestyle='--', color='black', label=f'Best threshold: {best_threshold:.2f}')                                                                
  plt.xlabel('Threshold')                                                                                                                                                  
  plt.title('Precision, Recall and F1 vs Threshold')                                                                                                                       
  plt.legend()                                                                                                                                                             
  plt.show()     

In [ ]:
  # Problem: SMOTE resulted in an unusually high optimal threshold (0.98)                                                                                                  
  # This means the model only flags fraud when it is 98% confident — too strict                                                                                            
  # Root cause: SMOTE generated too many synthetic fraud samples (394 -> 227,451)                                                                                          
  # causing the model to become overly conservative on the real test set                                                                                                   
  #                                                                                                                                                                        
  # Solution: Try class_weight instead of SMOTE                                                                                                                            
  # Instead of generating synthetic samples, we tell XGBoost to penalize                                                                                                   
  # missed fraud cases more heavily during training                                                                                                                        
  # scale_pos_weight = normal_count / fraud_count = 227451 / 394 = ~577                                                                                                    
                                                                            

In [ ]:
  # Class Weight Model                                                                                                                                                     
  # Instead of SMOTE, we use scale_pos_weight to handle class imbalance
  # XGBoost penalizes missed fraud cases 577x more than missed normal cases                                                                                                
                                                                                                                                                                           
  model_cw = XGBClassifier(                                                                                                                                                
      random_state=42,                                                                                                                                                     
      eval_metric='logloss',                                                                                                                                               
      scale_pos_weight=227451/394
  )                                                                                                                                                                        
  model_cw.fit(X_train, y_train)
                                                                                                                                                                           
  y_pred_cw = model_cw.predict(X_test)
  print(classification_report(y_test, y_pred_cw, target_names=['Normal', 'Fraud']))

In [ ]:
  # Final Model Comparison:                                                                                                                                                
  # Base (no balancing):  Precision=0.92, Recall=0.81, F1=0.86                                                                                                             
  # SMOTE:                Precision=0.73, Recall=0.89, F1=0.80 — too many false alarms                                                                                     
  # Class Weight:         Precision=0.88, Recall=0.85, F1=0.86 — best balance                                                                                              
  #                                                                                                                                                                        
  # Winner: Class Weight model (model_cw)                                                                                                                                  
  # Reason: Higher recall than base model, much better precision than SMOTE                                                                                                
  # No threshold issues — works at default 0.5                                                                                                                             
  # This model will be saved as model.pkl for production use                                                                                                               
                                                                 

In [ ]:
  # ROC-AUC Score                                                                                                                                                          
  # Measures the model's ability to distinguish fraud from normal transactions                                                                                             
  # Score ranges from 0.5 (random) to 1.0 (perfect)                                                                                                                        
                                                                                                                                                                           
  from sklearn.metrics import roc_auc_score, RocCurveDisplay                                                                                                               
                                                                                                                                                                           
  y_scores_cw = model_cw.predict_proba(X_test)[:, 1]                                                                                                                       
  auc_score = roc_auc_score(y_test, y_scores_cw)
  print(f"ROC-AUC Score: {auc_score:.4f}")                                                                                                                                 
                  
  RocCurveDisplay.from_predictions(y_test, y_scores_cw)                                                                                                                    
  plt.title('ROC Curve - Class Weight Model')
  plt.show()       

In [ ]:
  # Decision: Upgrade to Stacking Ensemble                                                                                                                                 
  # XGBoost alone achieved: Precision=0.88, Recall=0.85, F1=0.86, ROC-AUC=0.9652                                                                                           
  # Research shows stacking (XGBoost + LightGBM + CatBoost) can reach ROC-AUC 0.98+                                                                                        
  # Next step: Train all three base models, then combine with Logistic Regression meta-model        

In [ ]:
pip install lightgbm catboost    

In [ ]:
  # Stacking Ensemble - XGBoost + LightGBM                                                                                                                                 
  # Meta-model: Logistic Regression combines both models' predictions for final decision                                                                                   
                                                                                                                                                                           
  from lightgbm import LGBMClassifier                                                                                                                                      
  from sklearn.linear_model import LogisticRegression                                                                                                                      
  from sklearn.ensemble import StackingClassifier                                                                                                                          
   
  base_models = [                                                                                                                                                          
      ('xgb', XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=227451/394)),
      ('lgbm', LGBMClassifier(random_state=42, class_weight='balanced', verbose=-1))                                                                                       
  ]
                                                                                                                                                                           
  meta_model = LogisticRegression()                                                                                                                                        
   
  stacking_model = StackingClassifier(                                                                                                                                     
      estimators=base_models,
      final_estimator=meta_model,
      cv=5
  )                                                                                                                                                                        
   
  stacking_model.fit(X_train, y_train)                                                                                                                                     
  print("Stacking model trained ✅")


In [ ]:
  # Stacking Model Evaluation                                                                                                                                              
  # Comparing stacking ensemble results against single XGBoost model                                                                                                       
                                                                                                                                                                           
  y_pred_stack = stacking_model.predict(X_test)                                                                                                                            
  print(classification_report(y_test, y_pred_stack, target_names=['Normal', 'Fraud']))
                                                                                                                                                                           
  y_scores_stack = stacking_model.predict_proba(X_test)[:, 1]
  auc_stack = roc_auc_score(y_test, y_scores_stack)                                                                                                                        
  print(f"ROC-AUC: {auc_stack:.4f}")       

In [ ]:
                                                                                                                                                                           
  # Final Model Selection: XGBoost with class_weight                                                                                                                       
  # Reason: Higher recall (0.85 vs 0.80) — catching more fraud is priority in banking                                                                                      
  # Stacking improved ROC-AUC and precision but sacrificed recall                                                                                                          
  # model_cw is saved as the production model                                                                                                                              
                                                                                                                                                                           
  import joblib                                                                                                                                                            
                                                                                                                                                                           
  joblib.dump(model_cw, 'model.pkl')                                                                                                                                       
  print("Model saved as model.pkl ✅")
                                           

In [ ]:
  # Feature Importance
  # Shows which features contributed most to fraud detection                                                                                                               
  # Helps understand what drives the model's decisions
                                                                                                                                                                           
  feature_importance = pd.Series(
      model_cw.feature_importances_,                                                                                                                                       
      index=X_train.columns
  ).sort_values(ascending=False).head(10)

  plt.figure(figsize=(10, 6))                                                                                                                                              
  feature_importance.plot(kind='bar', color='steelblue')
  plt.title('Top 10 Most Important Features')                                                                                                                              
  plt.xlabel('Feature')                                                                                                                                                    
  plt.ylabel('Importance Score')
  plt.xticks(rotation=45)                                                                                                                                                  
  plt.tight_layout()
  plt.show()

  print(feature_importance)  

In [3]:
# (Replaced by the correct fraud sample cell below — see next cell)


[406.0, -2.3122265423263, 1.95199201064158, -1.60985073229769, 3.9979055875468, -0.522187864667764, -1.42654531920595, -2.53738730624579, 1.39165724829804, -2.77008927719433, -2.77227214465915, 3.20203320709635, -2.89990738849473, -0.595221881324605, -4.28925378244217, 0.389724120274487, -1.14074717980657, -2.83005567450437, -0.0168224681808257, 0.416955705037907, 0.126910559061474, 0.517232370861764, -0.0350493686052974, -0.465211076182388, 0.320198198514526, 0.0445191674731724, 0.177839798284401, 0.261145002567677, -0.143275874698919, 0.0]


In [5]:
# Extract a real fraud sample with correct feature scaling
# This is the exact format expected by the /score API endpoint and the SQS consumer

import pandas as pd
import joblib

df_raw = pd.read_csv('creditcard.csv')
scaler_loaded = joblib.load('scaler.pkl')

# Scale Amount and Time using the saved production scaler (same parameters as training)
df_raw[['Amount_scaled', 'Time_scaled']] = scaler_loaded.transform(df_raw[['Amount', 'Time']])
df_raw = df_raw.drop(['Amount', 'Time'], axis=1)

fraud_sample = df_raw[df_raw['Class'] == 1].iloc[0].drop('Class')
print("Fraud sample features (30 values) — send this to /score endpoint:")
print(fraud_sample.tolist())


[-2.3122265423263, 1.95199201064158, -1.60985073229769, 3.9979055875468, -0.522187864667764, -1.42654531920595, -2.53738730624579, 1.39165724829804, -2.77008927719433, -2.77227214465915, 3.20203320709635, -2.89990738849473, -0.595221881324605, -4.28925378244217, 0.389724120274487, -1.14074717980657, -2.83005567450437, -0.0168224681808257, 0.416955705037907, 0.126910559061474, 0.517232370861764, -0.0350493686052974, -0.465211076182388, 0.320198198514526, 0.0445191674731724, 0.177839798284401, 0.261145002567677, -0.143275874698919, -0.35322939296682354, -1.9880335064229064]
